In [0]:
from pyspark.sql import functions as F

def _run(name):
    src  = f"/Volumes/supplychainiq/bronze/landing/{name}/"
    ckpt = f"/Volumes/supplychainiq/ops/checkpoints/bronze_{name}"
    tgt  = f"supplychainiq.bronze.{name}"
    before = spark.table(tgt).count() if spark.catalog.tableExists(tgt) else 0
    q = (spark.readStream.format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("header", "true")
         .option("cloudFiles.schemaLocation", f"{ckpt}/schema")
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
         .option("rescuedDataColumn", "_rescued_data")
         .load(src)
         .select("*",
                 F.col("_metadata.file_path").alias("_source_file"),
                 F.current_timestamp().alias("_ingested_at"))
         .writeStream
         .option("checkpointLocation", ckpt)
         .option("mergeSchema", "true")          # let the Delta table accept the new column
         .trigger(availableNow=True)
         .toTable(tgt))
    q.awaitTermination()
    after = spark.table(tgt).count()
    print(f"{name}: {after - before:,} rows ingested (total {after:,})")

def ingest_csv(name, max_attempts=3):
    for attempt in range(1, max_attempts + 1):
        try:
            return _run(name)
        except Exception as e:
            msg = str(e)
            if "UNKNOWNFIELD" in msg.upper().replace("_", ""):
                print(f"{name}: attempt {attempt} stopped on a NEW COLUMN (expected with addNewColumns)")
                print("   ", msg[:300].replace("\n", " "))
                continue                         # restart picks up the evolved schema
            raise                                # any other error is a real failure

for name in ["shipments", "inventory"]:
    ingest_csv(name)

In [0]:
%sql
SELECT 'shipments' t, COUNT(*) n, MIN(arrival_date) mn, MAX(arrival_date) mx FROM supplychainiq.bronze.shipments
UNION ALL
SELECT 'inventory', COUNT(*), MIN(snapshot_date), MAX(snapshot_date) FROM supplychainiq.bronze.inventory;

In [0]:
%sql
SELECT store_id, COUNT(*) AS rows
FROM supplychainiq.bronze.pos_events
WHERE event_date = '2014-08-13'
GROUP BY store_id ORDER BY store_id;

In [0]:
%sql
SELECT arrival_date >= '2015-08-11' AS after_drift, COUNT(*) AS rows,
       COUNT(carrier_code) AS has_carrier, COUNT(_rescued_data) AS rescued,
       MIN(arrival_date), MAX(arrival_date)
FROM supplychainiq.bronze.shipments GROUP BY 1 ORDER BY 1;

In [0]:
%sql
LIST '/Volumes/supplychainiq/ops/checkpoints/bronze_shipments/schema/_schemas/';


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS supplychainiq.ops.ground_truth;